# 실습 1주차: 펭귄의 몸무게를 맞히는 첫 모형

> **시나리오 — 오늘 만들 것**
>
>
> 펭귄 **333마리의 부리·날개 치수**로 **몸무게(g)** 를 맞히는 모형을 만든다.
>
> $$\text{부리 길이},\ \text{부리 깊이},\ \text{날개 길이} \;\longrightarrow\; \boxed{\text{모형}} \;\longrightarrow\; \text{몸무게}$$
>
> **데이터 열기 → $X,y$ 만들기 → 예측 계산 → 손실 측정 → 학습 → 러닝커브 → 평가**
> 까지 오늘 한 번에 끝까지 간다. 필요한 numpy·PyTorch 문법은 그때그때 배운다.
>
> - **대응 이론**: [Ch01 들어가기: 데이터와 모형](ch01.qmd)
> - 코드는 완성되어 있다. 중간중간 **직접 해보기** 칸은 스스로 채운 뒤 아래 정답과 맞춰 본다.
> - 채점하지 않는다.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
print('torch', torch.__version__)

---

# 1. 데이터 열기

## 1-1. 인터넷에서 바로 불러온다

In [ ]:
URL = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv'
df = pd.read_csv(URL)
df.head()

## 1-2. 무엇이 들어 있나

In [ ]:
print('행, 열 :', df.shape)
print()
print(df.dtypes)

숫자 열(`float64`)과 글자 열(`object`)이 섞여 있다.
Ch01에서 말한 **정형 데이터** — 행이 개체, 열이 변수다.

## 1-3. 빈칸이 있는 행은 뺀다

In [ ]:
print('제거 전 :', len(df))
print('열별 결측 수:\n', df.isna().sum())

d = df.dropna().reset_index(drop=True)
print('\n제거 후 :', len(d))

## 1-4. 그림으로 먼저 본다

In [ ]:
plt.figure(figsize=(5.5, 3.8))
plt.scatter(d['flipper_length_mm'], d['body_mass_g'], s=14, alpha=0.6)
plt.xlabel('flipper length (mm)'); plt.ylabel('body mass (g)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

날개가 길수록 무겁다. **직선 하나로 꽤 설명될 것 같다** — 오늘 만들 모형이 그 직선이다.

> **직접 해보기 ① — 다른 변수로 그려 보기**
>
>
> `bill_length_mm` 와 `body_mass_g` 의 산점도를 그려 보시오.
> 날개 길이만큼 뚜렷한 관계인가?

In [ ]:
# ✏️ 직접 채워 보세요
plt.figure(figsize=(5.5, 3.8))
plt.scatter(...)          # ← 여기를 채우세요
plt.xlabel('bill length (mm)'); plt.ylabel('body mass (g)')
plt.grid(alpha=0.3); plt.show()

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
plt.figure(figsize=(5.5, 3.8))
plt.scatter(d['bill_length_mm'], d['body_mass_g'], s=14, alpha=0.6, color='C1')
plt.xlabel('bill length (mm)'); plt.ylabel('body mass (g)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

관계가 있긴 하지만 **훨씬 흩어져 있다.** 변수마다 예측에 주는 도움이 다르다.

---

# 2. $X$ 와 $y$ 만들기

모형에 넣으려면 표를 **숫자 배열 두 개**로 갈라야 한다.

## 2-1. 입력과 정답을 나눈다

In [ ]:
cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']

X = d[cols].to_numpy(dtype='float32')       # 입력 (독립변수)
y = d['body_mass_g'].to_numpy(dtype='float32')  # 정답 (종속변수)

print('X.shape :', X.shape, '   ← (n, p) = (개체 수, 변수 수)')
print('y.shape :', y.shape, '   ← (n,)')
print('\nX 첫 3줄:\n', X[:3])
print('\ny 첫 3개:', y[:3])

> **`(n, p)` — 앞으로 계속 나오는 모양**
>
>
> - **`n`** = 데이터 포인트 수 (펭귄 333마리)
> - **`p`** = 변수 수 (치수 3개)
>
> `X[i]` 는 **한 마리**, `X[:, j]` 는 **한 변수 전체**다.

In [ ]:
print('0번 펭귄     X[0]   :', X[0], '  y[0] :', y[0])
print('날개 길이 전체 X[:, 2] :', X[:5, 2], '...')

> **직접 해보기 ② — 변수 2개짜리 입력 만들기**
>
>
> 부리 길이와 날개 길이 **두 개만** 쓰는 입력 `X2` 를 만드시오. shape은 `(333, 2)` 여야 한다.

In [ ]:
# ✏️ 직접 채워 보세요
X2 = None        # ← 여기를 채우세요

assert X2 is not None, '아직 채우지 않았습니다'
assert X2.shape == (333, 2), f'shape이 다릅니다: {X2.shape}'
print('통과 ', X2[:2])

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
X2 = d[['bill_length_mm', 'flipper_length_mm']].to_numpy(dtype='float32')
assert X2.shape == (333, 2), f'shape이 다릅니다: {X2.shape}'
print('통과 ', X2[:2])

---

# 3. 모형이 하는 일 — 예측 한 줄

Ch01의 다중회귀 그대로다.

$$\hat{y} = w_1 x_1 + w_2 x_2 + w_3 x_3 + b$$

## 3-1. 먼저 한 마리

In [ ]:
w = np.array([4.0, 10.0, 50.0], dtype='float32')   # 아무 값이나 넣어 본다
b = np.float32(-6000.0)

x0 = X[0]
pred0 = (w * x0).sum() + b          # 곱해서 더한다 = 내적

print('입력  :', x0)
print('가중치:', w)
print('예측  :', round(float(pred0), 1), 'g')
print('실제  :', y[0], 'g')

## 3-2. 전부 한 번에 — 행렬 곱

333마리를 반복문으로 돌 필요가 없다. **행렬 곱 한 번**이면 끝난다.

In [ ]:
yhat = X @ w + b            # (n, p) @ (p,) → (n,)

print('X.shape    :', X.shape)
print('w.shape    :', w.shape)
print('yhat.shape :', yhat.shape)
print('\n앞 5마리 예측:', yhat[:5].round(1))
print('앞 5마리 실제:', y[:5])

In [ ]:
# 반복문으로 해도 결과가 같은지 확인
loop = np.array([(w * X[i]).sum() + b for i in range(len(X))])
print('반복문과 같은가:', np.allclose(loop, yhat))

> **행렬 곱의 모양 규칙**
>
>
> $$(n,\ \color{red}{p}) \;@\; (\color{red}{p},) \;\longrightarrow\; (n,)$$
>
> **안쪽 두 숫자가 같아야** 곱해진다. 딥러닝 코드에서 나는 에러의 대부분이 이 규칙 위반이다.
> 막히면 `.shape` 을 찍어 본다.


> **직접 해보기 ③ — 가중치를 바꿔 예측해 보기**
>
>
> 날개 길이의 가중치만 `50 → 20` 으로 줄인 `w_small` 로 예측하시오.
> 예측 몸무게가 어떻게 변하는가?

In [ ]:
# ✏️ 직접 채워 보세요
w_small = None            # ← 여기를 채우세요
yhat_small = None         # ← 여기도

assert yhat_small is not None and yhat_small.shape == (333,)
print('원래 w  평균 예측:', round(float(yhat.mean()), 1))
print('작은 w  평균 예측:', round(float(yhat_small.mean()), 1))

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
w_small = np.array([4.0, 10.0, 20.0], dtype='float32')
yhat_small = X @ w_small + b
print('원래 w  평균 예측:', round(float(yhat.mean()), 1))
print('작은 w  평균 예측:', round(float(yhat_small.mean()), 1))
print('실제    평균     :', round(float(y.mean()), 1))

---

# 4. 얼마나 틀렸나 — 손실

## 4-1. MSE 한 줄

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

In [ ]:
err = y - yhat
mse = (err ** 2).mean()

print('오차 앞 5개 :', err[:5].round(1))
print('MSE         :', round(float(mse), 1))
print('RMSE        :', round(float(np.sqrt(mse)), 1), 'g   ← 평균적으로 이만큼 틀린다')

RMSE는 MSE의 제곱근이다. **단위가 원래대로(g) 돌아와** 해석하기 쉽다.

> **직접 해보기 ④ — 손실 함수를 직접 만들기**
>
>
> `y` 와 `yhat` 을 받아 **RMSE** 를 돌려주는 함수 `rmse(y_true, y_pred)` 를 작성하시오.

In [ ]:
# ✏️ 직접 채워 보세요
def rmse(y_true, y_pred):
    return None            # ← 여기를 채우세요

got = rmse(y, yhat)
assert got is not None, '아직 채우지 않았습니다'
assert abs(got - 434.1) < 1.0, f'값이 이상합니다: {got}'
print('통과  RMSE =', round(float(got), 1))

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
def rmse(y_true, y_pred):
    return np.sqrt(((y_true - y_pred) ** 2).mean())

print('RMSE =', round(float(rmse(y, yhat)), 1))

## 4-2. 가중치를 바꾸면 손실이 어떻게 변하나

날개 길이의 가중치 하나만 움직여 본다.

In [ ]:
candidates = np.linspace(0, 100, 101)
losses = [rmse(y, X @ np.array([4.0, 10.0, c], dtype='float32') + b) for c in candidates]

best = candidates[int(np.argmin(losses))]
plt.figure(figsize=(5.5, 3.6))
plt.plot(candidates, losses)
plt.axvline(best, color='red', ls='--', label=f'best = {best:.0f}')
plt.xlabel('weight of flipper_length'); plt.ylabel('RMSE (g)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print('가장 좋은 값:', best, '   그때 RMSE:', round(float(min(losses)), 1))

**곡선의 바닥을 찾는 것** — 이것이 학습이다.
가중치가 3개면 3차원, 신경망에서는 수백만 차원이 된다. 손으로 찾을 수 없으니
컴퓨터에게 시킨다.

---

# 5. 학습 — 컴퓨터가 가중치를 찾게 한다

## 5-1. 숫자 크기를 맞춰 준다

날개 길이는 200 언저리, 부리 깊이는 17 언저리다. 크기가 제각각이면 학습이 잘 안 된다.
**평균 0, 표준편차 1** 로 맞춰 준다.

In [ ]:
mu, sd = X.mean(axis=0), X.std(axis=0)
ym, ys = y.mean(), y.std()

Xs = (X - mu) / sd
ys_ = (y - ym) / ys

print('맞추기 전 평균:', X.mean(axis=0).round(2))
print('맞춘 뒤   평균:', Xs.mean(axis=0).round(4))
print('맞춘 뒤   표준편차:', Xs.std(axis=0).round(4))

> 왜 이렇게 하는지는 **4주차**에 제대로 배운다. 지금은 "크기를 맞춰 주면 잘 배운다"만 기억한다.

## 5-2. PyTorch 텐서로 바꾼다

In [ ]:
Xt = torch.tensor(Xs)
yt = torch.tensor(ys_).unsqueeze(1)      # (n,) → (n, 1) 세로로 세운다

print('Xt :', tuple(Xt.shape), Xt.dtype)
print('yt :', tuple(yt.shape), yt.dtype)

## 5-3. 모형은 한 줄

In [ ]:
model = nn.Linear(in_features=3, out_features=1)

print(model)
print('가중치 w :', model.weight.detach().numpy().round(4), tuple(model.weight.shape))
print('편향   b :', model.bias.detach().numpy().round(4))
print('파라미터 수:', sum(p.numel() for p in model.parameters()), ' = 3 + 1')

`nn.Linear(3, 1)` 이 곧 $\hat{y} = w_1x_1 + w_2x_2 + w_3x_3 + b$ 다.
지금은 값이 **무작위**다 — 아직 아무것도 배우지 않았다.

## 5-4. 학습시킨다

In [ ]:
def fit(model, X, y, epochs=200, lr=0.1, X_val=None, y_val=None):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    hist = {'train': [], 'val': []}
    for ep in range(epochs):
        optimizer.zero_grad()          # ① 기울기 초기화
        loss = criterion(model(X), y)  # ② 예측하고 손실 계산
        loss.backward()                # ③ 기울기 구하기
        optimizer.step()               # ④ 가중치 갱신
        hist['train'].append(loss.item())
        if X_val is not None:
            with torch.no_grad():
                hist['val'].append(criterion(model(X_val), y_val).item())
    return hist

> 이 함수 안의 **네 줄**(①~④)이 딥러닝 학습의 전부다.
> **3주차에 이 함수를 직접 만든다.** 오늘은 그냥 쓴다.

In [ ]:
torch.manual_seed(42)
model = nn.Linear(3, 1)
hist = fit(model, Xt, yt, epochs=500, lr=0.1)

print('첫 손실  :', round(hist['train'][0], 4))
print('마지막 손실:', round(hist['train'][-1], 4))

## 5-5. 러닝커브 — 학습이 되고 있나

In [ ]:
plt.figure(figsize=(5.8, 3.6))
plt.plot(hist['train'])
plt.xlabel('epoch'); plt.ylabel('MSE (standardized)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

> **러닝커브는 매번 그린다**
>
>
> 손실이 **내려가면** 배우고 있는 것이고, **평평하면** 다 배웠거나 학습률이 잘못된 것이다.
> 숫자 하나만 보지 말고 **곡선을 본다** — 이 습관을 15주 내내 쓴다.


## 5-6. 무엇을 배웠나

In [ ]:
w_learned = model.weight.detach().numpy()[0] * ys / sd     # 원래 단위로 되돌린다
b_learned = float(model.bias) * ys + ym - (w_learned * mu).sum()

for c, wv in zip(cols, w_learned):
    print(f'{c:20s} {wv:8.2f} g / mm')
print(f'{"절편":20s} {b_learned:8.1f} g')

**날개 길이 1mm가 몸무게 약 50g** 에 해당한다고 모형이 판단했다.
파라미터는 그냥 숫자가 아니라 **읽을 수 있는 값**이다.

이 문제는 수학적으로 정답을 바로 구할 수도 있다. 경사하강법이 그 답에 도달했는지 대조해 본다.

In [ ]:
from sklearn.linear_model import LinearRegression
exact = LinearRegression().fit(X, y)
print('경사하강법 :', w_learned.round(2), round(b_learned, 1))
print('수학적 정답:', exact.coef_.round(2), round(float(exact.intercept_), 1))

> **같은 답에 도달했다.** 선형회귀는 정답 공식이 있어서 대조가 가능하지만,
> 신경망은 그런 공식이 없다 — 그래서 앞으로는 **경사하강법이 유일한 길**이 된다.


## 5-7. 예측이 맞는지 눈으로

In [ ]:
with torch.no_grad():
    pred = model(Xt).squeeze(1).numpy() * ys + ym       # 원래 단위(g)로

print('RMSE :', round(float(rmse(y, pred)), 1), 'g')

plt.figure(figsize=(4.4, 4.4))
plt.scatter(y, pred, s=14, alpha=0.6)
lim = [y.min() - 200, y.max() + 200]
plt.plot(lim, lim, 'r--', lw=1.5)
plt.xlim(lim); plt.ylim(lim)
plt.xlabel('actual (g)'); plt.ylabel('predicted (g)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

점이 빨간 대각선에 가까울수록 잘 맞힌 것이다.

> **직접 해보기 ⑤ — 학습률을 바꿔 보기**
>
>
> 학습률을 `0.001`, `0.1`, `1.5` 로 바꿔 각각 200에폭 학습시키고 러닝커브를 한 그림에 겹쳐 보시오.
> 어느 것이 가장 빨리 내려가는가? 너무 크면 어떻게 되는가?

In [ ]:
# ✏️ 직접 채워 보세요
plt.figure(figsize=(6, 3.6))
for lr in [0.001, 0.1, 1.5]:
    torch.manual_seed(42)
    m = nn.Linear(3, 1)
    h = ...                    # ← fit(...)을 호출하세요
    plt.plot(h['train'], label=f'lr = {lr}')
plt.yscale('log'); plt.xlabel('epoch'); plt.ylabel('MSE')
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
plt.figure(figsize=(6, 3.6))
for lr in [0.001, 0.1, 1.5]:
    torch.manual_seed(42)
    m = nn.Linear(3, 1)
    h = fit(m, Xt, yt, epochs=200, lr=lr)
    plt.plot(h['train'], label=f'lr = {lr}')
plt.yscale('log'); plt.xlabel('epoch'); plt.ylabel('MSE')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

학습률은 사람이 정하는 값 — Ch01에서 말한 **하이퍼파라미터**다.

---

# 6. 속지 않으려면 — 훈련과 시험을 나눈다

지금까지는 **배운 데이터로 성적을 매겼다.** 시험 문제를 미리 보고 푼 셈이다.

## 6-1. 나눈다

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print('훈련 :', X_tr.shape[0], '마리')
print('시험 :', X_te.shape[0], '마리')

> **`random_state` 는 반드시 지정한다**
>
>
> 지정하지 않으면 실행할 때마다 다르게 나뉘어 **성능 비교가 불가능**해진다.


## 6-2. 훈련 데이터로만 배우고, 시험 데이터로 평가한다

In [ ]:
mu, sd = X_tr.mean(0), X_tr.std(0)          # 통계도 훈련 데이터에서만 구한다
ym, ys = y_tr.mean(), y_tr.std()

def to_tensor(A, b):
    return (torch.tensor((A - mu) / sd),
            torch.tensor((b - ym) / ys).unsqueeze(1))

Xt_tr, yt_tr = to_tensor(X_tr, y_tr)
Xt_te, yt_te = to_tensor(X_te, y_te)

torch.manual_seed(42)
model = nn.Linear(3, 1)
hist = fit(model, Xt_tr, yt_tr, epochs=200, lr=0.1, X_val=Xt_te, y_val=yt_te)

plt.figure(figsize=(5.8, 3.6))
plt.plot(hist['train'], label='train')
plt.plot(hist['val'], label='test')
plt.xlabel('epoch'); plt.ylabel('MSE'); plt.legend()
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
with torch.no_grad():
    p_tr = model(Xt_tr).squeeze(1).numpy() * ys + ym
    p_te = model(Xt_te).squeeze(1).numpy() * ys + ym

print('훈련 RMSE :', round(float(rmse(y_tr, p_tr)), 1), 'g')
print('시험 RMSE :', round(float(rmse(y_te, p_te)), 1), 'g')

## 6-3. 나누지 않으면 어떤 착각을 하게 되는가

일부러 아주 유연한 모형(변수를 잔뜩 만든 것)을 훈련시켜 본다.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# 데이터가 적을수록 과적합이 잘 보인다 — 훈련 30마리만 쓴다
X_small, y_small = X_tr[:30], y_tr[:30]

rows = []
for deg in [1, 2, 3, 4, 5]:
    pf = PolynomialFeatures(deg)
    A_tr, A_te = pf.fit_transform(X_small), pf.transform(X_te)
    lin = LinearRegression().fit(A_tr, y_small)
    rows.append({'차수': deg, '변수 수': A_tr.shape[1],
                 '훈련 RMSE': round(float(rmse(y_small, lin.predict(A_tr))), 1),
                 '시험 RMSE': round(float(rmse(y_te, lin.predict(A_te))), 1)})
res = pd.DataFrame(rows)
print(res.to_string(index=False))

In [ ]:
plt.figure(figsize=(5.8, 3.6))
plt.plot(res['차수'], res['훈련 RMSE'], 'o-', label='train')
plt.plot(res['차수'], res['시험 RMSE'], 'o-', label='test')
plt.yscale('log'); plt.xlabel('polynomial degree'); plt.ylabel('RMSE (g), log scale')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

> **훈련 성적만 보면 정반대의 결론을 내린다**
>
>
> 차수를 올릴수록 **훈련 RMSE는 계속 좋아진다.** 하지만 **시험 RMSE는 어느 지점부터 나빠진다.**
> 훈련 성적만 봤다면 가장 복잡한 모형을 골랐을 것이다 — 그것이 **과적합**이다.
>
> **이 수업의 규칙: 학습에 쓴 데이터로 성능을 재지 않는다.**


---

# 7. 완성 — 처음부터 끝까지 한 번에

지금까지 한 것을 **한 덩어리**로 이어 붙인다. 앞으로 모든 실습이 이 뼈대를 반복한다.

In [ ]:
def run(cols, epochs=300, lr=0.1, seed=42):
    # ① 데이터
    X = d[cols].to_numpy(dtype='float32')
    y = d['body_mass_g'].to_numpy(dtype='float32')

    # ② 나눈다
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

    # ③ 크기를 맞춘다 (통계는 훈련에서만)
    mu, sd = X_tr.mean(0), X_tr.std(0)
    ym, ys = y_tr.mean(), y_tr.std()
    T = lambda A, b: (torch.tensor((A - mu) / sd),
                      torch.tensor((b - ym) / ys).unsqueeze(1))
    Xt_tr, yt_tr = T(X_tr, y_tr)
    Xt_te, yt_te = T(X_te, y_te)

    # ④ 모형
    torch.manual_seed(seed)
    model = nn.Linear(len(cols), 1)

    # ⑤ 학습
    hist = fit(model, Xt_tr, yt_tr, epochs=epochs, lr=lr, X_val=Xt_te, y_val=yt_te)

    # ⑥ 평가 (원래 단위로 되돌려 보고한다)
    with torch.no_grad():
        p_te = model(Xt_te).squeeze(1).numpy() * ys + ym
    return model, hist, float(rmse(y_te, p_te)), (y_te, p_te)


model, hist, test_rmse, (y_true, y_pred) = run(cols)
print('사용한 변수 :', cols)
print('테스트 RMSE :', round(test_rmse, 1), 'g')
print('몸무게 표준편차:', round(float(y.std()), 1), 'g  ← 아무것도 안 하면 이만큼 틀린다')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].plot(hist['train'], label='train'); axes[0].plot(hist['val'], label='test')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('MSE'); axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].scatter(y_true, y_pred, s=16, alpha=0.65)
lim = [y_true.min() - 200, y_true.max() + 200]
axes[1].plot(lim, lim, 'r--', lw=1.5)
axes[1].set_xlim(lim); axes[1].set_ylim(lim)
axes[1].set_xlabel('actual (g)'); axes[1].set_ylabel('predicted (g)')
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

> **직접 해보기 ⑥ — 변수 조합을 바꿔 비교하기**
>
>
> `run()` 에 넣는 변수 조합을 바꿔 가며 테스트 RMSE를 비교하시오.
>
> 1. `['flipper_length_mm']` — 날개 길이 하나만
> 2. `['bill_length_mm', 'bill_depth_mm']` — 부리만
> 3. `['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']` — 전부
>
> 어느 변수가 가장 큰 역할을 하는가?

In [ ]:
# ✏️ 직접 채워 보세요
for c in [...]:                       # ← 위 세 조합을 넣으세요
    _, _, r, _ = run(c)
    print(f'{str(c):55s} 테스트 RMSE {r:7.1f} g')

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
for c in [['flipper_length_mm'],
          ['bill_length_mm', 'bill_depth_mm'],
          ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']]:
    _, _, r, _ = run(c)
    print(f'{str(c):55s} 테스트 RMSE {r:7.1f} g')

날개 길이 하나만으로도 상당 부분이 설명되고, 부리 치수를 더하면 조금 더 좋아진다.
**어떤 변수를 넣을 것인가**도 모형 설계의 일부다.

---

# 8. 정리

> **오늘 만든 파이프라인**
>
>
> $$\text{데이터} \to X, y \to \text{분할} \to \text{크기 맞추기} \to \text{모형} \to \text{학습} \to \text{러닝커브} \to \text{평가}$$
>
> | 하고 싶은 일 | 코드 |
> |------|------|
> | 데이터 불러오기 | `pd.read_csv(URL)` |
> | 결측 제거 | `df.dropna()` |
> | $X$ 만들기 | `d[cols].to_numpy(dtype='float32')` → `(n, p)` |
> | 예측 한 번에 | `X @ w + b` |
> | 손실 | `((y - yhat)**2).mean()` |
> | 분할 | `train_test_split(X, y, test_size=0.2, random_state=42)` |
> | 텐서로 | `torch.tensor(...)`, 세로로 세우기 `.unsqueeze(1)` |
> | 모형 | `nn.Linear(p, 1)` |
> | 학습 4줄 | `zero_grad → loss → backward → step` |
> | 파라미터 보기 | `model.weight`, `model.bias` |


## 스스로 확인해 보기

아래 결과를 먼저 예상한 뒤 실행해서 대조한다.

In [ ]:
A = np.arange(12, dtype='float32').reshape(3, 4)
v = np.array([1., 0., 2., 0.], dtype='float32')

print('A =\n', A)
print('\nA.shape       :', A.shape)
print('A[0]          :', A[0])
print('A[:, 1]       :', A[:, 1])
print('A.mean(axis=0):', A.mean(axis=0))
print('A @ v         :', A @ v, ' shape', (A @ v).shape)

lin = nn.Linear(4, 1)
print('\nnn.Linear(4,1) 파라미터 수:', sum(p.numel() for p in lin.parameters()))
print('출력 shape:', tuple(lin(torch.tensor(A)).shape))

---

## 다음 실습

[실습 2주차: 퍼셉트론을 쌓아 신경망 만들기](lab02.qmd) —
오늘의 `nn.Linear(3, 1)` 을 **여러 개, 여러 층**으로 늘린다.
직선으로 안 되던 문제가 풀리기 시작한다.